# Fine-tuning NLLB-200-distilled-600M — bribri ↔ español

Notebook del **Avance 2** del Proyecto Integrador (Equipo 66, Tec de Monterrey).

Ejecuta el fine-tuning del corpus `corpus_v0` sobre NLLB-200-distilled-600M y calcula spBLEU, chrF y chrF++. Pensá para correr en **Colab con GPU** (Runtime → Change runtime type → T4 / L4 / A100).

**Tiempo estimado en T4**: ~30-45 min para 3 épocas con batch 8 sobre 1.505 pares.

## 1. Setup — clonar repo e instalar dependencias

In [ ]:
import subprocess, sys, os, pathlib

REPO_URL = "https://github.com/dleiva98/proyecto-integrador.git"
BRANCH = "claude/integrator-training-metrics-q45Cz"
REPO_DIR = pathlib.Path("/content/proyecto-integrador")

if not REPO_DIR.exists():
    subprocess.check_call(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)])
os.chdir(REPO_DIR)
print("cwd:", os.getcwd())

In [ ]:
%pip install -q torch "transformers==4.48.3" tqdm sacrebleu pydantic polars pyarrow
import torch
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu :", torch.cuda.get_device_name(0))

## 2. Splits
Generamos `data/splits/{train,val,test}.jsonl` si no existen.

In [ ]:
from pathlib import Path
splits = Path("data/splits")
if not (splits / "train.jsonl").exists():
    subprocess.check_call([sys.executable, "scripts/make_splits.py"])
else:
    print("splits ya existen")

for s in ["train", "val", "test"]:
    p = splits / f"{s}.jsonl"
    print(s, sum(1 for _ in p.open()), "pares")

## 3. Entrenamiento

In [ ]:
sys.path.insert(0, str(Path("src").resolve()))
from voces_corpus.training.nllb_train import TrainConfig, train

cfg = TrainConfig(
    epochs=3,
    batch_size=8,
    lr=5e-4,
    eval_every=100,
    log_every=20,
    use_float16=True,
    output_dir=Path("outputs"),
)
history = train(cfg, repo_root=Path.cwd())

## 4. Evaluación final sobre TEST

In [ ]:
import json
from torch.utils.data import DataLoader
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, DataCollatorForSeq2Seq
from voces_corpus.training.nllb_train import TranslationDataset, evaluate_split, _read_jsonl

device = "cuda" if torch.cuda.is_available() else "cpu"
model = AutoModelForSeq2SeqLM.from_pretrained("outputs/final_nllb").to(device)
tokenizer = AutoTokenizer.from_pretrained("outputs/final_nllb")

test_data = _read_jsonl(Path("data/splits/test.jsonl"))
collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, padding="longest", max_length=cfg.max_length, pad_to_multiple_of=8)

def make_test_loader(src_lang, tgt_lang, src_tok, tgt_tok):
    ds = TranslationDataset(test_data, src_lang, tgt_lang, src_tok, tgt_tok, False, tokenizer, cfg.max_length)
    return DataLoader(ds, batch_size=cfg.batch_size, shuffle=False, collate_fn=collator)

test_s2t = make_test_loader(cfg.src_lang, cfg.tgt_lang, cfg.src_lang_token, cfg.tgt_lang_token)
test_t2s = make_test_loader(cfg.tgt_lang, cfg.src_lang, cfg.tgt_lang_token, cfg.src_lang_token)

res_s2t = evaluate_split(model, tokenizer, test_s2t, cfg.tgt_lang_token, device, cfg.max_length)
res_t2s = evaluate_split(model, tokenizer, test_t2s, cfg.src_lang_token, device, cfg.max_length)

summary = {
    "es->bri": {k: res_s2t[k] for k in ("eval_loss", "spbleu", "chrf", "chrfpp")},
    "bri->es": {k: res_t2s[k] for k in ("eval_loss", "spbleu", "chrf", "chrfpp")},
    "avg": {
        "eval_loss": (res_s2t["eval_loss"] + res_t2s["eval_loss"]) / 2,
        "spbleu":    (res_s2t["spbleu"]    + res_t2s["spbleu"])    / 2,
        "chrf":      (res_s2t["chrf"]      + res_t2s["chrf"])      / 2,
        "chrfpp":    (res_s2t["chrfpp"]    + res_t2s["chrfpp"])    / 2,
    },
    "n_test": len(test_data),
}
print(json.dumps(summary, indent=2, ensure_ascii=False))

Path("outputs").mkdir(exist_ok=True)
Path("outputs/test_metrics.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False))

with open("outputs/test_predictions.jsonl", "w") as fh:
    for direction, res in (("es->bri", res_s2t), ("bri->es", res_t2s)):
        for pred, ref in zip(res["predictions"], res["references"]):
            fh.write(json.dumps({"direction": direction, "prediction": pred, "reference": ref}, ensure_ascii=False) + "\n")
print("\nGuardado: outputs/test_metrics.json y outputs/test_predictions.jsonl")

## 5. Curvas de entrenamiento

In [ ]:
import matplotlib.pyplot as plt
hist = json.loads(Path("outputs/metrics.json").read_text())
steps = hist["steps"]

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
axes[0].plot(steps, hist["src2tgt"]["loss"], label="es→bri")
axes[0].plot(steps, hist["tgt2src"]["loss"], label="bri→es")
axes[0].plot(steps, hist["avg"]["loss"], label="avg", linestyle="--")
axes[0].set_title("Val loss"); axes[0].set_xlabel("step"); axes[0].legend(); axes[0].grid(True)

axes[1].plot(steps, hist["src2tgt"]["spbleu"], label="es→bri")
axes[1].plot(steps, hist["tgt2src"]["spbleu"], label="bri→es")
axes[1].plot(steps, hist["avg"]["spbleu"], label="avg", linestyle="--")
axes[1].set_title("spBLEU"); axes[1].set_xlabel("step"); axes[1].legend(); axes[1].grid(True)

axes[2].plot(steps, hist["src2tgt"]["chrfpp"], label="es→bri")
axes[2].plot(steps, hist["tgt2src"]["chrfpp"], label="bri→es")
axes[2].plot(steps, hist["avg"]["chrfpp"], label="avg", linestyle="--")
axes[2].set_title("chrF++"); axes[2].set_xlabel("step"); axes[2].legend(); axes[2].grid(True)

plt.tight_layout()
plt.savefig("outputs/training_curves.png", dpi=120)
plt.show()

## 6. (Opcional) Guardar artefactos en Drive
Monta Drive y copia `outputs/` para no perder el modelo cuando la sesión termine.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p /content/drive/MyDrive/proyecto-integrador-outputs
# !cp -r outputs /content/drive/MyDrive/proyecto-integrador-outputs/

## 7. Subir métricas al repo (opcional)
Si querés commitear `outputs/test_metrics.json` y la curva de entrenamiento desde Colab:

```bash
!git config user.email "daniel.lk98@gmail.com"
!git config user.name  "dleiva98"
!git checkout claude/integrator-training-metrics-q45Cz
!git add outputs/test_metrics.json outputs/training_curves.png
!git commit -m "avance 2: métricas y curvas del fine-tuning NLLB"
!git push origin claude/integrator-training-metrics-q45Cz
```
(necesitás un PAT con permisos de escritura; seguí las indicaciones que muestre git al pedir credenciales.)